
# ETL sales data
muc tieu :
- Extract du lieu tu file csv
- Transform du lieu
- Load vao clickhouse

# Cau truc du an 
- file csv --> extract --> transform --> Store Data(ClickHouse STG/DIM/FACT)

In [166]:
#import thu vien 
import pandas as pd
import csv
import clickhouse_connect
from dotenv import load_dotenv
import os
load_dotenv()

from until.batch_insert import batch_insert

# Connect Database

In [167]:
#ket noi database
def get_client():
    return clickhouse_connect.get_client(
        host=os.getenv("HOST"),
        port=int(os.getenv("PORT")),
        username=os.getenv("CLICKHOUSE_USERNAME"),
        password=os.getenv("CLICKHOUSE_PASSWORD"),
        database=os.getenv("CLICKHOUSE_DATABASE")
    )
get_client()

# Extract Data && Raw data 

In [168]:

#DOC DU LIEU TU FILE CSV
def extract():
    files = {
        "customers": os.getenv("FILECUSTOMER"),
        "products": os.getenv("FILEPRODUCT"),
        "sellers": os.getenv("FILESELLER"),
        "orders": os.getenv("FILEORDER"),
        "order_items": os.getenv("FILEORDERITEM")
    }
    data = {}

    for table_name, file_path in files.items():
        df = pd.read_csv(file_path)

        print(f"\n===== {table_name} =====")
        display(df.head())
        print("Shape:", df.shape)
        print("\nNull values:")
        print(df.isnull().sum())
        print("\nduplicate values:")
        print(df.duplicated().sum())

        data[table_name] = df

    return data
extract()


===== customers =====


,customer_id,customer_name,gender,city,segment
0,C001,Nguyen Van A,Male,Hanoi,Regular
1,C002,Tran Thi B,Female,HCM,VIP
2,C003,Le Van C,M,Da Nang,Regular
3,C004,Pham Thi D,Female,Hanoi,VIP
4,C005,Hoang Van E,Nam,Hanoi,Regular


Shape: (14, 5)

Null values:
customer_id      0
customer_name    0
gender           1
city             0
segment          1
dtype: int64

duplicate values:
2

===== products =====


,product_id,product_name,category,brand
0,P001,iPhone 15,Mobile,Apple
1,P002,Galaxy S24,mobile,Samsung
2,P003,Macbook Air,Laptop,Apple
3,P004,Dell XPS,laptop,Dell
4,P005,AirPods,Accessory,Apple


Shape: (12, 4)

Null values:
product_id      0
product_name    0
category        0
brand           0
dtype: int64

duplicate values:
2

===== sellers =====


...

,order_id,customer_id,seller_id,order_date,ship_date,status
0,O001,C001,S001,2025-01-05,2025-01-07,Delivered
1,O002,C002,S002,2025/01/08,2025-01-10,Delivered
2,O003,C003,S003,2025-01-10,2025-01-15,Delivered
3,O004,C004,S001,10-01-2025,2025-01-12,Delivered
4,O005,C005,S004,2025-01-12,NaN,Pending


Shape: (12, 6)

Null values:
order_id       0
customer_id    0
seller_id      0
order_date     0
ship_date      2
status         0
dtype: int64

duplicate values:
1

===== order_items =====


,order_id,product_id,quantity,unit_price,discount
0,O001,P001,1,25000000,1000000
1,O001,P005,2,5000000,0
2,O002,P002,1,22000000,500000
3,O003,P003,1,30000000,0
4,O004,P004,1,28000000,1000000


Shape: (13, 5)

Null values:
order_id      0
product_id    0
quantity      0
unit_price    0
discount      0
dtype: int64

duplicate values:
1


{'customers':    customer_id   customer_name  gender       city  segment
 0         C001    Nguyen Van A    Male      Hanoi  Regular
 1         C002      Tran Thi B  Female        HCM      VIP
 2         C003        Le Van C       M    Da Nang  Regular
 3         C004      Pham Thi D  Female      Hanoi      VIP
 4         C005     Hoang Van E     Nam      Hanoi  Regular
 5         C005     Hoang Van E     Nam      Hanoi  Regular
 6         C006    Nguyen Van F     NaN  Hai Phong  Regular
 7         C007      Tran Van G    male        HCM      VIP
 8         C008        Le Thi H       F    Da Nang      NaN
 9         C009      Pham Van I  Female      Hanoi      VIP
 10        C010     Hoang Thi K      Nữ        HCM  Regular
 11        C001    Nguyen Van A    Male      Hanoi  Regular
 12        C011    Nguyen Van L    Male    Ha Noi   Regular
 13        C012  Nguyen Thanh L  Female     Korea       VIP,
 'products':    product_id    product_name     category     brand
 0        P001      

# Transform Data

In [169]:
#lam sach products
def clean_products(rows):
    df_products = pd.DataFrame(rows)
    df_products["category"] = (
        df_products["category"]
        .str.strip()
        .str.title()
        .fillna("Unknown"))
    df_products["brand"] = (
        df_products["brand"]
        .str.strip()
        .str.title()
        .fillna("Unknown"))

    df_products = df_products.drop_duplicates(subset=["product_id"])
    print("cleaned products")
    return df_products.to_dict("records")


In [170]:
# lam sach sellers
def clean_sellers(rows):
    city_map = {
        "hanoi": "Hanoi",
        "ha noi": "Hanoi",
        "hcm": "Ho Chi Minh",
        "ho chi minh": "Ho Chi Minh",
        "da nang": "Da Nang",
        "hai phong": "Hai Phong"
    }
    df_seller= pd.DataFrame(rows)

    df_seller["seller_name"] =( 
        df_seller["seller_name"]
        .str.strip()
        .str.title()
        .fillna("Unknown")
    )
    df_seller["city"] = (
        df_seller["city"]
        .str.strip()
        .str.lower()
        .map(city_map)
        .fillna("Unknown")
    )

    df_seller = df_seller.drop_duplicates(subset=["seller_id"])

    return df_seller.to_dict("records") 

In [171]:
#lam sach customer
def clean_customers(rows):
    gender_map = {
        "m": "Male", "male": "Male",
        "f": "Female", "female": "Female",
        "nam": "Male", "nu": "Female", "nữ": "Female"
    }

    city_map = {
        "hanoi": "Hanoi",
        "ha noi": "Hanoi",
        "hcm": "Ho Chi Minh",
        "ho chi minh": "Ho Chi Minh",
        "da nang": "Da Nang",
        "hai phong": "Hai Phong"
    }
    df_customer = pd.DataFrame(rows)
    df_customer["customer_name"] =(
        df_customer["customer_name"]
        .str.strip()
        .str.title()
        .fillna("Unknown")
        ) 
    df_customer["gender"] =(
        df_customer["gender"]
        .str.strip()
        .str.lower()
        .map(gender_map)
        .fillna("Unknown")
        ) 
    df_customer["city"] = (
        df_customer["city"]
        .str.strip()
        .str.lower()
        .map(city_map)
        .fillna("Unknown")
        )
    df_customer["segment"] = (
        df_customer["segment"]
        .str.strip()
        .str.title()
        .fillna("Regular")
        )

    df_customer = df_customer.drop_duplicates(subset=["customer_id"])

    return df_customer.to_dict("records")

In [172]:
#lam sach orders_items
def clean_order_items(rows):
    df_order_items = pd.DataFrame(rows)
    df_order_items["quantity"] = df_order_items["quantity"].fillna(0).astype(int)
    df_order_items["unit_price"] = df_order_items["unit_price"].fillna(0).astype(float)
    df_order_items["discount"] = df_order_items["discount"].fillna(0).astype(float)

    df_order_items = df_order_items.drop_duplicates(subset=["order_id"])

    return df_order_items.to_dict("records")

In [173]:
#lam sach orders
def clean_orders(rows):
    df_order = pd.DataFrame(rows)
    #parse date
    df_order["order_date"]= df_order["order_date"].str.replace("/", "-")
    df_order["ship_date"]= df_order["ship_date"].str.replace("/", "-")

    df_order["order_date"] =pd.to_datetime(
        df_order["order_date"],
        format='%Y-%m-%d',
        errors='coerce'
        )
    df_order["ship_date"] = pd.to_datetime(
        df_order["ship_date"], 
        format='%Y-%m-%d',
        errors='coerce'
        )

    #loai bo cac dong co order_date hoac ship_date la NaT
    df_order = df_order.dropna(subset=["order_date", "ship_date"]) 

    df_order = df_order.drop_duplicates(subset=["order_id"])

    return df_order.to_dict("records")

# Load data to Database

In [174]:
#LOAD VAO BANG STG
def load_stg_table(client,df, table_name):
    df = pd.DataFrame(df)
    batch_insert(client,table_name, df)
    print(f"Loaded STG: {table_name}")


In [175]:
#LOAD VAO BANG DIM 
# dim_customer
def load_dim_customer(client,stg_customers):
    df_customer = pd.DataFrame(stg_customers).copy()
    df_customer = df_customer [
        [
            "customer_id",
            "customer_name", 
            "gender", 
            "city", 
            "segment"
        ]
    ]
    #chi lay du lieu chua co
    df_existing = client.query_df("select customer_id from dim_customer")
    if "customer_id" in df_existing.columns:
        df_customer = df_customer[
            ~df_customer["customer_id"].isin(
                df_existing["customer_id"]
            )
        ]
    if df_customer.empty:
        print("khong co customer moi")
        return []
    
    #tim trong bang dim check da ton tai customer_sk chua
    existing_sk = client.query_df("select customer_sk from dim_customer ")

    if existing_sk.empty:
        start_sk=1
    #neu co tim giatri max
    else:
        start_sk = existing_sk["customer_sk"].max() +1

    df_customer.insert(0,"customer_sk", range(start_sk , start_sk+ len(df_customer)))
    
    #insert theo batch( ketnoi clickhouse, ten bang, dataframe)
    batch_insert(client,"dim_customer",df_customer)
    print("Loaded DIM Customer")
    return df_customer.to_dict("records")

In [176]:

#dim_product
def load_dim_product(client,stg_products):
    df_products = pd.DataFrame(stg_products).copy()
    df_products = df_products[
        [
            "product_id",
            "product_name",
            "category",
            "brand"
        ]
    ]
    #chi lay du lieu chua co
    df_new = client.query_df("select product_id from dim_product")
    if "product_id" in df_new.columns:
        df_products = df_products[
            ~df_products["product_id"].isin(
                df_new["product_id"]
            )
        ]
    if df_products.empty:
        print("khong co product moi")
        return []
    
    #tim trong bang dim check da ton tai product_sk chua
    existing_df = client.query_df("select product_sk from dim_product ")

    if existing_df.empty:
        start_sk=1
    #neu co tim giatri max
    else:
        start_sk = existing_df["product_sk"].max() +1
    df_products.insert(0,"product_sk", range(start_sk , start_sk+ len(df_products)))

    batch_insert(client , "dim_product", df_products)

    print("Loaded DIM products")
    return df_products.to_dict("records")



In [177]:
#dim_seller
def load_dim_seller(client,stg_sellers):
    df_seller = pd.DataFrame(stg_sellers).copy()
    df_seller = df_seller[
        [
            "seller_id",
            "seller_name",
            "city"
        ]
    ]
    #chi lay du lieu chua co
    df_new = client.query_df("select seller_id from dim_seller")
    if "seller_id" in df_new.columns:
        df_seller = df_seller[
            ~df_seller["seller_id"].isin(
                df_new["seller_id"]
            )
        ]
    if df_seller.empty:
        print("khong co seller moi")
        return []
    
    #tim trong bang dim check da ton tai seller_sk chua
    existing_df = client.query_df("select seller_sk from dim_seller ")

    if existing_df.empty:
        start_sk=1
    #neu co tim giatri max
    else:
        start_sk = existing_df["seller_sk"].max() +1
    df_seller.insert(0,"seller_sk", range(start_sk , start_sk+ len(df_seller)))

    batch_insert(client,"dim_seller",df_seller)

    print("Loaded DIM seller")
    return df_seller.to_dict("records")

In [178]:
#dim_date
def load_dim_date(client,stg_orders):
    df_date = pd.DataFrame(stg_orders).copy()

    #tao dim_date voi cac cot year, month, day tu order_date
    dim_date =pd.DataFrame({
        "date_sk": df_date["order_date"].dt.strftime('%Y%m%d').astype("Int32"),
        "date": df_date["order_date"].dt.date,
        "year": df_date["order_date"].dt.year,
        "month": df_date["order_date"].dt.month,
        "day": df_date["order_date"].dt.day
        }
    )
    
    dim_date =dim_date.drop_duplicates(subset=["date_sk"])

    #chi lay du lieu chua co
    df_new = client.query_df("select date_sk from dim_date")
    if "date_sk" in df_new.columns:
        dim_date = dim_date[
            ~dim_date["date_sk"].isin(
                df_new["date_sk"]
            )
        ]
    if dim_date.empty:
        print("khong co date moi")
        return []

    batch_insert(client, "dim_date", dim_date)

    print("Loaded DIM Date")
    return dim_date.to_dict("records")


In [179]:
# LOAD VAO BANG FACT(gop 2 bang order va order_item) 
def load_fact(client,stg):
    df_item = pd.DataFrame(stg["order_items"])
    df_order = pd.DataFrame(stg["orders"])

    # merge 2 bang order_item va orders
    df_fact= df_item.merge(
        df_order[
            ["order_id","customer_id","order_date"]
        ],
        on="order_id",
        how="left"
    )

    #lấy dữ liệu từ db , vì neu khong lay o db thi khi chạy lần 2 mà khoogn có dữ liệu mới thì sẽ trả về là rỗng 
    df_dim_customer = client.query_df("select customer_id,customer_sk from dim_customer")
    df_dim_product = client.query_df("select product_sk, product_id from dim_product")

    #merge du lieu tu df_dim_product , giu lai tat ca du lieu cua df, lay product_sk cua bang df_dim_product
    df_fact =df_fact.merge(
        df_dim_product[
            ["product_id", "product_sk"]
        ],
        on="product_id",
        how="left"
    )
    #merge du lieu tu df_dim_customer
    df_fact =df_fact.merge(
        df_dim_customer[
            ["customer_id", "customer_sk"]
        ],
        on="customer_id",
        how="left"
    )
    df_fact["date_sk"] = (
        pd.to_datetime(df_fact["order_date"])
        .dt.strftime("%Y%m%d")
        .astype("Int32")
    )

    df_fact =df_fact.dropna(subset=["product_sk","customer_sk"])
    df_fact["product_sk"] = df_fact["product_sk"].astype(int)
    df_fact["customer_sk"] = df_fact["customer_sk"].astype(int)

    df_fact["revenue"] = df_fact["quantity"] * df_fact["unit_price"] - df_fact["discount"]
    df_fact["revenue"] = df_fact["revenue"].round(2).astype(float)

    df_fact = df_fact[
        [
            "order_id",
            "date_sk",
            "customer_sk",
            "product_sk",
            "quantity",
            "unit_price",
            "discount",
            "revenue"
        ]
    ]
    
    #chi lay du lieu moi
    df_new = client.query_df("select order_id from fact_order")
    if "order_id" in df_new.columns:
        df_fact = df_fact[
            ~df_fact["order_id"].isin(
                df_new["order_id"]
            )
        ]
    if df_fact.empty:
        print("Khong co fact moi")
        return


    batch_insert(client, "fact_order", df_fact)


# PIPELINE ETL && Data Cleaned

In [180]:
from config.database import get_client
from extract.extract_csv import extract
from transform.clean_data import (
    clean_customers, 
    clean_products, 
    clean_sellers, 
    clean_orders, 
    clean_order_items
)
from store_data.store_data import(
    load_stg_table,
    load_dim_customer,
    load_dim_date,
    load_dim_product,
    load_dim_seller,
    load_fact
)
def run_pipeline():
    print("\n START PIPELINE\n")

    #EXTRACT
    raw_data = extract()
    print("Extract done")

    #TRANSFORM
    customers = clean_customers(raw_data["customers"])
    products = clean_products(raw_data["products"])
    sellers = clean_sellers(raw_data["sellers"])
    orders = clean_orders(raw_data["orders"])
    order_items = clean_order_items(raw_data["order_items"])
    print("Transform done")

    client = get_client()

    # LOAD DATA VAO BANG STG 
    try:
        load_stg_table(client,raw_data["customers"], "stg_customer")
        load_stg_table(client,raw_data["products"], "stg_product")
        load_stg_table(client,raw_data["sellers"], "stg_seller")
        load_stg_table(client,raw_data["orders"], "stg_order")
        load_stg_table(client,raw_data["order_items"], "stg_order_item")
        print("STG loaded")

        #LOAD VAO BANG DIM
        load_dim_customer(client,customers)
        load_dim_product(client,products)
        load_dim_seller(client,sellers)
        load_dim_date(client,orders)
        print("DIM loaded")

        stg = {
            "customers": customers,
            "products": products,
            "sellers": sellers,
            "orders": orders,
            "order_items": order_items
        }
        #LOAD VAO BANG FACT
        load_fact(client,stg)
        print("FACT loaded")

        print("\n ETL xong\n")
        
    # dong ket noi db
    finally :
        client.close()
    
    table = {
        "df_customers" :pd.DataFrame(customers),
        "df_products":pd.DataFrame(products),
        "df_sellers" :pd.DataFrame(sellers),
        "df_orders" :pd.DataFrame(orders),
        "df_order_items" :pd.DataFrame(order_items)
    }

    for table_name, df in table.items():

        print(f"\n===== {table_name} =====")
        display(df.head())
        print("Shape:", df.shape)
        print("\nNull values:")
        print(df.isnull().sum())
        print("\nduplicate values:")
        print(df.duplicated().sum())

run_pipeline()


 START PIPELINE

Extract done
Transform done


Loaded STG: stg_customer
Loaded STG: stg_product
Loaded STG: stg_seller
Loaded STG: stg_order
Loaded STG: stg_order_item
STG loaded
khong co customer moi
khong co product moi
khong co seller moi
khong co date moi
DIM loaded
Khong co fact moi
FACT loaded

 ETL xong


===== df_customers =====


,customer_id,customer_name,gender,city,segment
0,C001,Nguyen Van A,Male,Hanoi,Regular
1,C002,Tran Thi B,Female,Ho Chi Minh,Vip
2,C003,Le Van C,Male,Da Nang,Regular
3,C004,Pham Thi D,Female,Hanoi,Vip
4,C005,Hoang Van E,Male,Hanoi,Regular


Shape: (12, 5)

Null values:
customer_id      0
customer_name    0
gender           0
city             0
segment          0
dtype: int64

duplicate values:
0

===== df_products =====


,product_id,product_name,category,brand
0,P001,iPhone 15,Mobile,Apple
1,P002,Galaxy S24,Mobile,Samsung
2,P003,Macbook Air,Laptop,Apple
3,P004,Dell XPS,Laptop,Dell
4,P005,AirPods,Accessory,Apple


...

Shape: (8, 3)

Null values:
seller_id      0
seller_name    0
city           0
dtype: int64

duplicate values:
0

===== df_orders =====


,order_id,customer_id,seller_id,order_date,ship_date,status
0,O001,C001,S001,2025-01-05,2025-01-07,Delivered
1,O002,C002,S002,2025-01-08,2025-01-10,Delivered
2,O003,C003,S003,2025-01-10,2025-01-15,Delivered
3,O006,C006,S005,2025-01-15,2025-01-20,Delivered
4,O007,C007,S006,2025-01-18,2025-01-22,Cancelled


Shape: (9, 6)

Null values:
order_id       0
customer_id    0
seller_id      0
order_date     0
ship_date      0
status         0
dtype: int64

duplicate values:
0

===== df_order_items =====


,order_id,product_id,quantity,unit_price,discount
0,O001,P001,1,25000000.0,1000000.0
1,O002,P002,1,22000000.0,500000.0
2,O003,P003,1,30000000.0,0.0
3,O004,P004,1,28000000.0,1000000.0
4,O005,P007,1,18000000.0,0.0


Shape: (11, 5)

Null values:
order_id      0
product_id    0
quantity      0
unit_price    0
discount      0
dtype: int64

duplicate values:
0


# Kết luận

Quy trình ETL đã thực hiện:

1. Extract dữ liệu từ CSV
2. Làm sạch dữ liệu
3. Xây dựng Staging Layer
4. Tạo Dimension Tables
5. Tạo Fact Table
6. Load vào ClickHouse
7. Validation dữ liệu
8. Phân tích KPI cơ bản

Data Warehouse đã sẵn sàng cho:
- Superset
- Dashboard Analytics